# 09 – Hybrid Multimodal Router Training (with W&B + Checkpointing)

This notebook trains a **hybrid VLM router** that combines:

- The **training loop, checkpointing, and W&B logging style** from your `08_training_router_with_images.ipynb`.
- A **new cross-modal transformer architecture** inspired by `06b_training_router_v2.ipynb`.

Architecture:

1. Frozen **CLIP vision encoder** → visual tokens
2. Frozen **BERT text encoder** → text tokens
3. Project both into a shared `d_model` and build a joint sequence:
   `CLS_router + vision_tokens + text_tokens`
4. Run through a stack of Transformer encoder layers (cross-attention over all tokens)
5. Classification head on the `CLS_router` token → router logits over VLMs
6. Train with **class-imbalance aware CE + KL soft-label loss**

Assumptions (same as 08):

- Trainer parquet files: `router_train_trainer.parquet`, `router_val_trainer.parquet`, `router_test_trainer.parquet`.
- Columns:
  - `router_best_model_id` (int, 0..num_models-1)
  - `router_soft_p_<model_name>` (soft label probabilities per class)
  - `prompt_raw`, `txt_prompt_length_words`, `txt_prompt_length_chars`, `img_width`, `img_height`
  - `image_png` (bytes) or `image_path` (path to image on disk)


In [1]:
import os
import io
import math
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Optional, Dict

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModel,
    CLIPVisionModel,
    CLIPImageProcessor,
    get_linear_schedule_with_warmup,
)

import matplotlib.pyplot as plt

try:
    import wandb
except ImportError:
    wandb = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


/home/hice1/vchopra37/scratch/projects/vlm_router/vlm_router_env/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Using device: cuda


In [2]:
@dataclass
class RouterConfig:
    # Paths
    project_root: Path = Path.cwd().parent  # adjust if needed
    data_root: Path = Path.cwd().parent.parent.parent / "dataset" / "final_dataset"
    router_subdir: str = "router_lexico"  # where trainer parquet files live
    output_dir: Path = Path.cwd() / "saved_output"/"router_training_outputs_hybrid"
    checkpoint_dir: Path = Path.cwd() / "saved_output"/"router_checkpoints_hybrid"
        # Optional: where to save training history JSON
    training_history_path: Optional[Path] = Path.cwd() / "saved_output"/"router_hybrid_training_history.json"


    # Trainer parquet filenames (one per split)
    train_file: str = "router_train_trainer.parquet"
    val_file: str   = "router_val_trainer.parquet"
    test_file: str  = "router_test_trainer.parquet"

    # Optional: image_root only used as fallback if image_png bytes missing
    image_root: Path = Path.cwd().parent.parent.parent / "dataset" / "which_vlm_data" / "images"

    # Model + tokenizer
    vision_encoder_name: str = "openai/clip-vit-base-patch32"
    text_encoder_name: str   = "bert-base-uncased"
    text_tokenizer_name: str = "bert-base-uncased"

    # Hybrid transformer architecture
    d_model: int = 330
    num_layers: int = 2
    num_heads: int = 6
    ffn_dim: int = 1536
    dropout: float = 0.1
    max_text_length: int = 256
    max_vision_tokens: int = 64  # truncate visual tokens to this

    use_image: bool = True
    freeze_vision: bool = True
    freeze_text_encoder: bool = True

    # Training hyperparameters (08-style)
    num_epochs: int = 15
    train_batch_size: int = 64
    eval_batch_size: int = 64
    learning_rate: float = 5e-5
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    warmup_ratio: float = 0.1

    # Loss configuration
    use_soft_labels: bool = True
    ce_weight: float = 0.7
    kl_weight: float = 0.3
    label_smoothing: float = 0.05
    imbalance_strategy: str = "class_weight"  # "none" or "class_weight"

    # W&B
    use_wandb: bool = True
    wandb_project: str = "vlm_router_hybrid"
    wandb_run_name: Optional[str] = "round_2"

    # Checkpointing
    checkpoint_every: int = 7
    resume_checkpoint: Optional[str] = None  # path to .pt checkpoint to resume from
    best_checkpoint_name: str = "best_model.pt"

    # Misc
    seed: int = 42
    num_workers: int = 4

    @property
    def device(self):
        return device

    def __post_init__(self):
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

    def to_dict(self):
        return asdict(self)

config = RouterConfig()

def set_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.seed)
print(config)
print("Using device:", config.device)


RouterConfig(project_root=PosixPath('/storage/ice1/1/0/vchopra37/projects/vlm_router/code_base/which_vlm'), data_root=PosixPath('/storage/ice1/1/0/vchopra37/projects/vlm_router/dataset/final_dataset'), router_subdir='router_lexico', output_dir=PosixPath('/storage/ice1/1/0/vchopra37/projects/vlm_router/code_base/which_vlm/artemis/saved_output/router_training_outputs_hybrid'), checkpoint_dir=PosixPath('/storage/ice1/1/0/vchopra37/projects/vlm_router/code_base/which_vlm/artemis/saved_output/router_checkpoints_hybrid'), training_history_path=PosixPath('/storage/ice1/1/0/vchopra37/projects/vlm_router/code_base/which_vlm/artemis/saved_output/router_hybrid_training_history.json'), train_file='router_train_trainer.parquet', val_file='router_val_trainer.parquet', test_file='router_test_trainer.parquet', image_root=PosixPath('/storage/ice1/1/0/vchopra37/projects/vlm_router/dataset/which_vlm_data/images'), vision_encoder_name='openai/clip-vit-base-patch32', text_encoder_name='bert-base-uncased', 

In [3]:
def build_router_text(row: pd.Series) -> str:
    """Construct the router text input from metadata + raw prompt.

    Mirrors the style used in your previous notebooks:
    lengths + image size + question.
    """
    prompt = str(row.get("prompt_raw", ""))
    len_words = row.get("txt_prompt_length_words", np.nan)
    len_chars = row.get("txt_prompt_length_chars", np.nan)
    img_w = row.get("img_width", np.nan)
    img_h = row.get("img_height", np.nan)

    parts = []
    if not pd.isna(len_words):
        parts.append(f"PromptLenWords: {int(len_words)}")
    if not pd.isna(len_chars):
        parts.append(f"PromptLenChars: {int(len_chars)}")
    if not pd.isna(img_w) and not pd.isna(img_h):
        parts.append(f"ImageWidth: {int(img_w)}")
        parts.append(f"ImageHeight: {int(img_h)}")

    meta_str = ", ".join(parts)
    if meta_str:
        return meta_str + " | Question: " + prompt
    else:
        return prompt

print("Example text construction (dummy row):")
dummy = pd.Series({
    "prompt_raw": "What is written on the sign?",
    "txt_prompt_length_words": 6,
    "txt_prompt_length_chars": 28,
    "img_width": 640,
    "img_height": 480,
})
print(build_router_text(dummy))


Example text construction (dummy row):
PromptLenWords: 6, PromptLenChars: 28, ImageWidth: 640, ImageHeight: 480 | Question: What is written on the sign?


In [4]:
class RouterDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        image_root: Path,
        image_processor: CLIPImageProcessor,
        tokenizer,
        config: RouterConfig,
        model_names: List[str],
    ):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.image_processor = image_processor
        self.tokenizer = tokenizer
        self.config = config
        self.model_names = model_names
        self.num_models = len(model_names)
        self.has_image_png = "image_png" in self.df.columns
        self.has_image_path = "image_path" in self.df.columns

    def __len__(self):
        return len(self.df)

    def _load_image(self, row: pd.Series) -> Optional[Image.Image]:
        if not self.config.use_image:
            return None
        img = None
        if self.has_image_png and pd.notna(row.get("image_png")):
            try:
                img_bytes = row["image_png"]
                if isinstance(img_bytes, (bytes, bytearray)):
                    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            except Exception:
                img = None
        if img is None and self.has_image_path and pd.notna(row.get("image_path")):
            img_path = row["image_path"]
            try:
                img = Image.open(img_path).convert("RGB")
            except Exception:
                img = None
        return img

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]

        text = build_router_text(row)
        enc = self.tokenizer(
            text,
            max_length=self.config.max_text_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        label = int(row["router_best_model_id"])
        soft_label_cols = [c for c in row.index if c.startswith("router_soft_p_")]
        soft_labels = None
        if soft_label_cols:
            soft = row[soft_label_cols].to_numpy(dtype=np.float32)
            soft_labels = torch.from_numpy(soft)

        pixel_values = None
        if self.config.use_image and self.image_processor is not None:
            img = self._load_image(row)
            if img is not None:
                proc = self.image_processor(images=img, return_tensors="pt")
                pixel_values = proc["pixel_values"].squeeze(0)

        sample = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "label": torch.tensor(label, dtype=torch.long),
        }
        if soft_labels is not None:
            sample["soft_labels"] = soft_labels
        if pixel_values is not None:
            sample["pixel_values"] = pixel_values
        return sample


def collate_batch(batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    input_ids = torch.stack([x["input_ids"] for x in batch], dim=0)
    attention_mask = torch.stack([x["attention_mask"] for x in batch], dim=0)
    labels = torch.stack([x["label"] for x in batch], dim=0)

    collated = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

    if "soft_labels" in batch[0]:
        soft_labels = torch.stack([x["soft_labels"] for x in batch], dim=0)
        collated["soft_labels"] = soft_labels

    if "pixel_values" in batch[0]:
        pixel_values = torch.stack([x["pixel_values"] for x in batch], dim=0)
        collated["pixel_values"] = pixel_values

    return collated


In [5]:
# ---- Hybrid Multimodal Router Model (frozen encoders + cross-modal transformer) ----
class MultimodalRouterModel(nn.Module):
    def __init__(
        self,
        config: RouterConfig,
        num_models: int,
        model_names: List[str],
    ):
        super().__init__()
        self.config = config
        self.num_models = num_models
        self.model_names = model_names

        # Vision encoder (CLIP vision tower)
        self.vision_encoder = CLIPVisionModel.from_pretrained(config.vision_encoder_name)
        d_vision = self.vision_encoder.config.hidden_size

        # Text encoder (BERT)
        self.text_encoder = AutoModel.from_pretrained(config.text_encoder_name)
        d_text = self.text_encoder.config.hidden_size

        if config.freeze_vision:
            for p in self.vision_encoder.parameters():
                p.requires_grad = False
        if config.freeze_text_encoder:
            for p in self.text_encoder.parameters():
                p.requires_grad = False

        # Project both into a shared d_model
        self.vision_proj = nn.Linear(d_vision, config.d_model)
        self.text_proj = nn.Linear(d_text, config.d_model)

        # CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, config.d_model))

        # Modality embedding: 0 = vision, 1 = text
        self.modality_embedding = nn.Embedding(2, config.d_model)

        # Positional embeddings: max length = 1 + max_vision_tokens + max_text_length
        max_seq_len = 1 + config.max_vision_tokens + config.max_text_length
        self.pos_embedding = nn.Parameter(torch.zeros(1, max_seq_len, config.d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.num_heads,
            dim_feedforward=config.ffn_dim,
            dropout=config.dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=config.num_layers)

        self.dropout = nn.Dropout(config.dropout)
        self.classifier = nn.Linear(config.d_model, num_models)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)

    def forward(self, pixel_values, input_ids, attention_mask):
        B = input_ids.size(0)

        # ----- Text encoder -----
        if self.config.freeze_text_encoder:
            with torch.no_grad():
                text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        else:
            text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_hidden = text_out.last_hidden_state  # [B, T, d_text]
        text_tokens = self.text_proj(text_hidden)  # [B, T, d_model]

        # ----- Vision encoder -----
        if self.config.use_image and pixel_values is not None:
            if self.config.freeze_vision:
                with torch.no_grad():
                    vision_out = self.vision_encoder(pixel_values=pixel_values)
            else:
                vision_out = self.vision_encoder(pixel_values=pixel_values)
            vision_hidden = vision_out.last_hidden_state  # [B, V, d_vision]
            vision_hidden = vision_hidden[:, : self.config.max_vision_tokens, :]
            vision_tokens = self.vision_proj(vision_hidden)  # [B, V, d_model]
        else:
            vision_tokens = torch.zeros(B, 0, self.config.d_model, device=input_ids.device)

        # ----- Build joint sequence CLS + V + T -----
        cls = self.cls_token.expand(B, 1, -1)  # [B, 1, d_model]
        seq = torch.cat([cls, vision_tokens, text_tokens], dim=1)  # [B, L, d_model]
        L = seq.size(1)

        # Modality ids: CLS + vision (0), text (1)
        num_vision = vision_tokens.size(1)
        num_text = text_tokens.size(1)
        mod_ids = torch.zeros(B, L, dtype=torch.long, device=seq.device)
        if num_text > 0:
            mod_ids[:, -num_text:] = 1
        mod_emb = self.modality_embedding(mod_ids)

        # Positional embeddings
        if L > self.pos_embedding.size(1):
            pos_emb = self.pos_embedding[:, :L, :]
        else:
            pos_emb = self.pos_embedding[:, :L, :]
        seq = seq + mod_emb + pos_emb
        seq = self.dropout(seq)

        # Padding mask for transformer: True = pad
        padding_mask = torch.zeros(B, L, dtype=torch.bool, device=seq.device)
        if num_text > 0:
            text_valid = attention_mask == 1
            padding_mask[:, -num_text:] = ~text_valid

        hidden = self.transformer(seq, src_key_padding_mask=padding_mask)
        cls_hidden = hidden[:, 0, :]
        logits = self.classifier(self.dropout(cls_hidden))
        return logits


In [6]:
# ---- Class weights helper ----
def build_class_weights(train_df: pd.DataFrame, num_models: int, config: RouterConfig, device: torch.device):
    """
    Build class weights tensor for imbalance, to be used globally as class_weights_tensor.
    """
    counts = train_df["router_best_model_id"].value_counts().sort_index()
    freqs = counts.reindex(range(num_models)).fillna(1.0).to_numpy(dtype=np.float32)
    freqs = torch.from_numpy(freqs)

    # Milder than 1/freq: use 1/sqrt(freq) and clamp
    raw_weights = 1.0 / torch.sqrt(freqs)
    raw_weights = raw_weights / raw_weights.mean()
    raw_weights = torch.clamp(raw_weights, max=3.0)
    print("Class weights:", raw_weights.tolist())
    return raw_weights.to(device)


# Global tensor (will be filled after loading train_df)
class_weights_tensor: Optional[torch.Tensor] = None


# ---- Loss + metrics helpers (with class weighting) ----
def compute_losses(
    logits: torch.Tensor,
    labels: torch.Tensor,
    soft_labels: Optional[torch.Tensor],
    config: RouterConfig,
):
    """
    Compute:
      - CE loss with optional class weighting + label smoothing
      - Optional KL loss to soft labels (router_soft_p_*)
      - Accuracy
    Returns a dict with keys: loss, loss_ce, (optional) loss_kl, acc.
    """
    global class_weights_tensor

    # --- Hard labels: cross-entropy with optional class weights ---
    if class_weights_tensor is not None:
        ce_loss_fn = nn.CrossEntropyLoss(
            weight=class_weights_tensor,
            label_smoothing=config.label_smoothing,
        )
    else:
        ce_loss_fn = nn.CrossEntropyLoss(
            label_smoothing=config.label_smoothing,
        )

    ce_loss = ce_loss_fn(logits, labels)
    loss = config.ce_weight * ce_loss
    loss_dict = {"loss": loss, "loss_ce": ce_loss}

    # --- Soft label KL-divergence (router_soft_p_*) ---
    if config.use_soft_labels and soft_labels is not None and soft_labels.numel() > 0:
        eps = 1e-8

        target_p = soft_labels.clamp(min=eps)
        target_p = target_p / target_p.sum(dim=-1, keepdim=True).clamp(min=eps)

        log_probs = torch.log_softmax(logits, dim=-1)
        kl = (target_p * (torch.log(target_p + eps) - log_probs)).sum(dim=-1)
        kl_loss = kl.mean()

        loss = loss + config.kl_weight * kl_loss
        loss_dict["loss"] = loss
        loss_dict["loss_kl"] = kl_loss

    # --- Accuracy ---
    preds = logits.argmax(dim=-1)
    acc = (preds == labels).float().mean()
    loss_dict["acc"] = acc

    return loss_dict



In [7]:



def train_one_epoch(model, loader, optimizer, scheduler, config: RouterConfig, epoch: int, use_wandb: bool = False):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    total_batches = 0

    for step, batch in enumerate(loader):
        pixel_values = batch.get("pixel_values", None)
        if pixel_values is not None:
            pixel_values = pixel_values.to(config.device)
        input_ids = batch["input_ids"].to(config.device)
        attention_mask = batch["attention_mask"].to(config.device)
        labels = batch["labels"].to(config.device)
        soft_labels = batch.get("soft_labels", None)
        if soft_labels is not None:
            soft_labels = soft_labels.to(config.device)

        optimizer.zero_grad()
        logits = model(pixel_values, input_ids, attention_mask)
        losses = compute_losses(logits, labels, soft_labels, config)
        loss = losses["loss"]
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
        optimizer.step()

        total_loss += loss.item()
        total_acc += losses["acc"].item()
        total_batches += 1

    avg_loss = total_loss / max(total_batches, 1)
    avg_acc  = total_acc / max(total_batches, 1)
    return avg_loss, avg_acc


def eval_one_epoch(model, loader, config: RouterConfig):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    total_batches = 0

    with torch.no_grad():
        for batch in loader:
            pixel_values = batch.get("pixel_values", None)
            if pixel_values is not None:
                pixel_values = pixel_values.to(config.device)
            input_ids = batch["input_ids"].to(config.device)
            attention_mask = batch["attention_mask"].to(config.device)
            labels = batch["labels"].to(config.device)
            soft_labels = batch.get("soft_labels", None)
            if soft_labels is not None:
                soft_labels = soft_labels.to(config.device)

            logits = model(pixel_values, input_ids, attention_mask)
            losses = compute_losses(logits, labels, soft_labels, config)

            total_loss += losses["loss"].item()
            total_acc += losses["acc"].item()
            total_batches += 1

    avg_loss = total_loss / max(total_batches, 1)
    avg_acc  = total_acc / max(total_batches, 1)
    return avg_loss, avg_acc


In [8]:
def save_checkpoint(path: Path, model: nn.Module, optimizer, scheduler, epoch: int, config: RouterConfig, best_val_acc: float):
    state = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "epoch": epoch,
        "config": config.to_dict(),
        "best_val_acc": best_val_acc,
    }
    torch.save(state, path)
    print(f"Saved checkpoint to {path}")


def load_checkpoint(path: Path, model: nn.Module, optimizer=None, scheduler=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and ckpt.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch = ckpt.get("epoch", 0) + 1
    best_val_acc = ckpt.get("best_val_acc", 0.0)
    print(f"Loaded checkpoint from {path}, resuming at epoch {start_epoch}")
    return start_epoch, best_val_acc


In [9]:
print("Loading trainer parquet datasets...")
router_dir = config.data_root / config.router_subdir
train_path = router_dir / config.train_file
val_path   = router_dir / config.val_file
test_path  = router_dir / config.test_file

train_df = pd.read_parquet(train_path)
val_df   = pd.read_parquet(val_path)
test_df  = pd.read_parquet(test_path)


Loading trainer parquet datasets...


In [10]:
# Class counts per label for sampler
class_counts = train_df["router_best_model_id"].value_counts().to_dict()
train_labels = train_df["router_best_model_id"].to_numpy()

sample_weights = np.array(
    [1.0 / class_counts[label] for label in train_labels],
    dtype=np.float32,
)
print("Class counts:", class_counts)

from torch.utils.data import WeightedRandomSampler

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),  # one epoch ~ dataset size
    replacement=True,
)



Class counts: {1: 34665, 2: 17771, 4: 8893, 0: 2020, 3: 614}


In [11]:

soft_label_cols = [c for c in train_df.columns if c.startswith("router_soft_p_")]
num_models = len(soft_label_cols)
assert num_models > 0, "No soft label columns (router_soft_p_*) found in train_df"
print("Detected", num_models, "models from soft label columns:")
print(soft_label_cols)

# Model names (for reference / plotting later)
model_names = [c.replace("router_soft_p_", "") for c in soft_label_cols]
print("Model names:", model_names)


Detected 5 models from soft label columns:
['router_soft_p_deepseek_ocr', 'router_soft_p_qwen2_5_vl_3b', 'router_soft_p_qwen2_5_vl_7b', 'router_soft_p_qwen3_vl_8b_thinking', 'router_soft_p_gemma_3_27b']
Model names: ['deepseek_ocr', 'qwen2_5_vl_3b', 'qwen2_5_vl_7b', 'qwen3_vl_8b_thinking', 'gemma_3_27b']


In [12]:


image_processor = CLIPImageProcessor.from_pretrained(config.vision_encoder_name) if config.use_image else None
tokenizer       = AutoTokenizer.from_pretrained(config.text_tokenizer_name)

train_dataset = RouterDataset(train_df, config.image_root, image_processor, tokenizer, config, model_names)
val_dataset   = RouterDataset(val_df,   config.image_root, image_processor, tokenizer, config, model_names)
test_dataset  = RouterDataset(test_df,  config.image_root, image_processor, tokenizer, config, model_names)


In [13]:

# train_loader = DataLoader(
#     train_dataset,
#     batch_size=config.train_batch_size,
#     shuffle=True,
#     num_workers=config.num_workers,
#     pin_memory=True,
#     collate_fn=collate_batch,
# )
train_loader = DataLoader(
    train_dataset,
    batch_size=config.train_batch_size,
    sampler=sampler,            # <- use sampler instead of shuffle
    num_workers=config.num_workers,
    pin_memory=True,
    collate_fn=collate_batch,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.eval_batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True,
    collate_fn=collate_batch,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=config.eval_batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True,
    collate_fn=collate_batch,
)

In [14]:


# --------- Class imbalance weights ---------
global class_weights_tensor
class_weights_tensor = build_class_weights(train_df, num_models, config, config.device)


# ============================================================
# ---- Initialize model, optimizer, auto-resume, W&B ----
# ============================================================
best_ckpt_path = config.checkpoint_dir / config.best_checkpoint_name

# Build model (hybrid cross-modal architecture)
model = MultimodalRouterModel(config, num_models=num_models, model_names=model_names)
model = model.to(config.device)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

# Optimizer (only trainable params)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)

Class weights: [1.2923405170440674, 0.31196582317352295, 0.4357092082500458, 2.3440585136413574, 0.6159259676933289]
Trainable parameters: 3523847


In [16]:
from torch.optim.lr_scheduler import ReduceLROnPlateau


# --------- Auto-resume from best checkpoint if it exists ---------
start_epoch   = 0
best_val_acc  = 0.0  # will get updated if we load a checkpoint
# After optimizer = AdamW(...)
scheduler = ReduceLROnPlateau(
    optimizer,
    mode="min",      # we want to minimize val loss
    factor=0.5,      # multiply LR by 0.5 when triggered
    patience=2,      # wait 2 epochs with no improvement
    min_lr=1e-6,     # do not go below this LR
    # verbose=True,    # print a message when LR changes
)


if best_ckpt_path.exists():
    ckpt = torch.load(best_ckpt_path, map_location=config.device, weight_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print(
        f" Found existing checkpoint at {best_ckpt_path}\n"
        f"   -> epoch={ckpt.get('epoch')}, best_val_acc={ckpt.get('best_val_acc', 0.0):.4f}"
    )

    # Try to restore optimizer as well (so LR, moments, etc. continue smoothly)
    if "optimizer_state_dict" in ckpt:
        try:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            print(" Loaded optimizer state from checkpoint.")
        except Exception as e:
            print(f"️ Could not load optimizer state, continuing with fresh optimizer: {e}")

    start_epoch  = ckpt.get("epoch", -1) + 1
    best_val_acc = ckpt.get("best_val_acc", 0.0)

    if start_epoch >= config.num_epochs:
        print(
            f"️ start_epoch ({start_epoch}) >= num_epochs ({config.num_epochs}); "
            "no further training epochs will run."
        )
else:
    print(f"No existing checkpoint at {best_ckpt_path}, training from scratch.")


No existing checkpoint at /storage/ice1/1/0/vchopra37/projects/vlm_router/code_base/which_vlm/artemis/saved_output/router_checkpoints_hybrid/best_model.pt, training from scratch.


In [17]:

# --------- W&B: safe logger + init ---------
def wandb_log_safe(data: dict):
    """Log to W&B but never crash the notebook if the backend dies."""
    if not config.use_wandb:
        return
    if wandb is None or wandb.run is None or getattr(wandb.run, "_is_finished", False):
        return
    try:
        wandb.log(data)
    except Exception as e:
        print(f"[WARN] W&B logging failed, skipping this log: {e}")
        # Optionally: permanently disable W&B for this run
        # config.use_wandb = False

run = None

# CosineAnnealingLR scheduler (epoch-based, like your 08 snippet)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.num_epochs,
)

# Some globals might already exist in your 08 notebook; if not, provide safe defaults
WANDB_AVAILABLE   = (wandb is not None)
DATA_UTILITY_SCHEME = globals().get("DATA_UTILITY_SCHEME", "unknown")
HIER_W_SAMPLE       = globals().get("HIER_W_SAMPLE", 0.7)
HIER_W_TASK         = globals().get("HIER_W_TASK", 0.2)
HIER_W_GLOBAL       = globals().get("HIER_W_GLOBAL", 0.1)

if config.use_wandb and WANDB_AVAILABLE:
    tags = [
        "router",
        "multimodal" if config.use_image else "text_only",
        f"utility:{DATA_UTILITY_SCHEME}",
        f"hier_w_sample:{HIER_W_SAMPLE}",
        f"hier_w_task:{HIER_W_TASK}",
        f"hier_w_global:{HIER_W_GLOBAL}",
    ]
    try:
        run = wandb.init(
            project=config.wandb_project,
            # entity=config.wandb_entity,  # add this in RouterConfig if you need entity
            name=config.wandb_run_name or "router_hybrid",
            config={
                **asdict(config),
                "data_utility_scheme": DATA_UTILITY_SCHEME,
                "data_hier_w_sample": HIER_W_SAMPLE,
                "data_hier_w_task": HIER_W_TASK,
                "data_hier_w_global": HIER_W_GLOBAL,
                "model_names": model_names,
            },
            tags=tags,
            settings=wandb.Settings(start_method="thread"),  # helps on clusters/Jupyter
        )
        wandb.watch(model, log="all", log_freq=100)
    except Exception as e:
        print(f"[WARN] W&B init failed, disabling logging: {e}")
        run = None
        config.use_wandb = False
else:
    config.use_wandb = False  # In case wandb isn't installed

# --------- Training history dict ---------
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}


wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.
wandb: Currently logged in as: vedaangchopra (vedaangchopra_gatech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
import json

# --------- Training loop (using new init + JSON history) ---------
for epoch in range(start_epoch, config.num_epochs):
    print(f"\n===== Epoch {epoch+1}/{config.num_epochs} =====")

    # 1) Train for one epoch
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        config,
        epoch,
        use_wandb=config.use_wandb,
    )

    # 2) Evaluate on validation set
    val_loss, val_acc = eval_one_epoch(
        model,
        val_loader,
        config,
    )
    scheduler.step(val_loss)

    # 3) Update in-memory history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Train: loss={train_loss:.4f}, acc={train_acc:.4f}")
    print(f"Val:   loss={val_loss:.4f}, acc={val_acc:.4f}")

    # 4) Safe W&B logging
    wandb_log_safe({
        "train/epoch_loss": train_loss,
        "train/epoch_acc": train_acc,
        "val/epoch_loss": val_loss,
        "val/epoch_acc": val_acc,
        "epoch": epoch + 1,
    })

    # 5) Save JSON history to disk (optional but nice to have)
    history_file = getattr(config, "training_history_path", None)
    if history_file is not None:
        try:
            with open(history_file, "w") as f:
                json.dump(history, f, indent=2)
        except Exception as e:
            print(f"[WARN] Failed to write training history JSON: {e}")

    # 6) Save regular checkpoint
    ckpt_path = config.checkpoint_dir / f"checkpoint_epoch_{epoch+1}.pt"
    save_checkpoint(ckpt_path, model, optimizer, scheduler, epoch, config, best_val_acc)

    # 7) Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(best_ckpt_path, model, optimizer, scheduler, epoch, config, best_val_acc)
        print(f" New best val acc: {best_val_acc:.4f} (epoch {epoch+1})")

print("\nTraining complete. Best val acc:", best_val_acc)



===== Epoch 1/15 =====


/home/hice1/vchopra37/scratch/projects/vlm_router/vlm_router_env/lib/python3.12/site-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
/home/hice1/vchopra37/scratch/projects/vlm_router/vlm_router_env/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:204: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unabl

Train: loss=0.7976, acc=0.4822
Val:   loss=1.1766, acc=0.3680
Saved checkpoint to /storage/ice1/1/0/vchopra37/projects/vlm_router/code_base/which_vlm/artemis/saved_output/router_checkpoints_hybrid/checkpoint_epoch_1.pt
Saved checkpoint to /storage/ice1/1/0/vchopra37/projects/vlm_router/code_base/which_vlm/artemis/saved_output/router_checkpoints_hybrid/best_model.pt
🎉 New best val acc: 0.3680 (epoch 1)

===== Epoch 2/15 =====
Train: loss=0.6726, acc=0.5993
Val:   loss=1.1669, acc=0.4190
Saved checkpoint to /storage/ice1/1/0/vchopra37/projects/vlm_router/code_base/which_vlm/artemis/saved_output/router_checkpoints_hybrid/checkpoint_epoch_2.pt
Saved checkpoint to /storage/ice1/1/0/vchopra37/projects/vlm_router/code_base/which_vlm/artemis/saved_output/router_checkpoints_hybrid/best_model.pt
🎉 New best val acc: 0.4190 (epoch 2)

===== Epoch 3/15 =====
Train: loss=0.6280, acc=0.6448
Val:   loss=1.0905, acc=0.4870
Saved checkpoint to /storage/ice1/1/0/vchopra37/projects/vlm_router/code_base/wh

In [ ]:
# Load best checkpoint (if exists) and evaluate on test set
if best_ckpt_path.is_file():
    print("Loading best checkpoint for test evaluation:", best_ckpt_path)
    # load_checkpoint returns (start_epoch, best_val_acc), we can ignore them here
    load_checkpoint(best_ckpt_path, model)
else:
    print("Best checkpoint not found, using last model state for test eval.")

# eval_one_epoch no longer takes class_weights; it uses global class_weights_tensor via compute_losses
test_loss, test_acc = eval_one_epoch(
    model,
    test_loader,
    config,
)
print(f"Test: loss={test_loss:.4f}, acc={test_acc:.4f}")

if config.use_wandb and run is not None:
    wandb.log({
        "test/loss": test_loss,
        "test/acc": test_acc,
    })


In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(10, 4))
plt.plot(epochs, history["train_loss"], label="Train loss")
plt.plot(epochs, history["val_loss"], label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss curves (Hybrid router)")
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(epochs, history["train_acc"], label="Train acc")
plt.plot(epochs, history["val_acc"], label="Val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy curves (Hybrid router)")
plt.show()


In [ ]:
if config.use_wandb and run is not None:
    run.finish()
    print("W&B run finished.")
